# EDA — EEG por Estímulo Emocional

In [ ]:
import ray, os, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RUTA       = ("/Volumes/Externo4T/GoogleDrive/Tesistas Neurodatos/"
              "Datos  Crudos/EEG/EEG por Estímulo/Estímulos Emocionales")
FS         = 125
SAT_THRESH = 90_000   # µV — valor rail OpenBCI ≈ ±187500
DEAD_THRESH = 10      # % de muestras en rail para declarar canal muerto

ray.init("ray://localhost:10001")
print(ray.cluster_resources())

In [ ]:
@ray.remote
def listar(ruta):
    import os, re
    return sorted(f for f in os.listdir(ruta)
                  if re.match(r"P\d+_Estimulo_\d+\.csv", f))

@ray.remote
def procesar(ruta, nombre, sat_thresh, dead_thresh, fs):
    """
    Pipeline por trial:
      1. Leer CSV sin header (nombres asignados por posición)
      2. Botar fila 0 (rail ±187500 en todos los canales)
      3. Detectar canales muertos: >dead_thresh% muestras en rail → excluir
      4. Band-pass 1–45 Hz con filtfilt (zero-phase) solo en canales buenos
      5. Stats + PSD sobre canales buenos filtrados
    """
    import os, re, numpy as np, pandas as pd
    from scipy.signal import butter, filtfilt

    path = os.path.join(ruta, nombre)
    m    = re.match(r"P(\d+)_Estimulo_(\d+)\.csv", nombre)
    pid, est = int(m.group(1)), int(m.group(2))

    def bp(sig, fs, lo=1.0, hi=45.0, orden=4):
        nyq = fs / 2
        b, a = butter(orden, [lo / nyq, hi / nyq], btype="band")
        return filtfilt(b, a, sig)

    def welch_np(sig, fs, nperseg):
        window = np.hanning(nperseg)
        hop    = nperseg // 2
        freqs  = np.fft.rfftfreq(nperseg, d=1.0 / fs)
        scale  = fs * (window ** 2).sum()
        segs, i = [], 0
        while i + nperseg <= len(sig):
            chunk = sig[i:i + nperseg] * window
            segs.append(np.abs(np.fft.rfft(chunk)) ** 2 / scale)
            i += hop
        return freqs, np.mean(segs, axis=0) if segs else (freqs, np.zeros_like(freqs))

    try:
        # — Lectura —
        with open(path, "r", errors="replace") as f:
            primera = f.readline()
        n_cols  = len(primera.strip().split(","))
        n_eeg   = min(16, n_cols - 2)
        n_extra = n_cols - 2 - n_eeg
        nombres  = (["SampleIndex"] +
                    [f"EXG Channel {i}" for i in range(n_eeg)] +
                    [f"Extra{i}" for i in range(n_extra)] +
                    ["Estimulo"])
        eeg_cols = [f"EXG Channel {i}" for i in range(n_eeg)]

        df = pd.read_csv(path, header=None, names=nombres, skipinitialspace=True)

        # — Paso 1: botar fila 0 anómala —
        df = df.iloc[1:].reset_index(drop=True)

        # — Paso 2: detectar canales muertos (ANTES de filtrar) —
        pct_rail = {
            c: float((pd.to_numeric(df[c], errors="coerce").abs() > sat_thresh).mean() * 100)
            for c in eeg_cols
        }
        muertos = [c for c in eeg_cols if pct_rail[c] > dead_thresh]
        buenos  = [c for c in eeg_cols if pct_rail[c] <= dead_thresh]
        n_dead  = len(muertos)

        # — Paso 3: band-pass 1–45 Hz zero-phase solo en canales buenos —
        min_len  = fs * 4
        filtradas = {}
        for c in buenos:
            raw = pd.to_numeric(df[c], errors="coerce").fillna(0).values.astype(float)
            filtradas[c] = bp(raw, fs) if len(raw) >= min_len else raw

        mean_amp = float(np.mean([s.mean() for s in filtradas.values()])) if filtradas else float("nan")

        # — Paso 4: PSD sobre señal filtrada —
        nperseg = fs * 4
        psds, f_arr = [], None
        for c, sig in filtradas.items():
            if len(sig) >= nperseg:
                f_arr, p = welch_np(sig, fs, nperseg)
                psds.append(p)

        return {
            "participante": pid, "estimulo": est,
            "n_muestras": len(df), "duracion_s": len(df) / fs,
            "n_ch_total": n_eeg,
            "n_dead_ch": n_dead,
            "n_buenos_ch": len(buenos),
            "canales_muertos": muertos,
            "mean_amp": mean_amp,
            "freqs": f_arr.tolist() if psds else None,
            "psd":   np.mean(psds, axis=0).tolist() if psds else None,
        }
    except Exception as e:
        return {"participante": pid, "estimulo": est, "error": str(e)}

In [ ]:
from ray.util.scheduling_strategies import NodeAffinitySchedulingStrategy

@ray.remote
def check_nodo(ruta):
    import os, ray
    return {
        "node_id": ray.get_runtime_context().get_node_id(),
        "tiene_disco": os.path.exists(ruta),
    }

# Probar cada nodo explícitamente por IP
recursos = ray.cluster_resources()
ips = [k.split("node:")[1] for k in recursos
       if k.startswith("node:") and "__internal_head__" not in k]
print("Nodos en el cluster:", ips)

checks = ray.get([
    check_nodo.options(resources={f"node:{ip}": 0.001}).remote(RUTA)
    for ip in ips
])
for ip, c in zip(ips, checks):
    print(f"  {ip} → disco: {c['tiene_disco']}")

nodo_disco = next((c["node_id"] for c in checks if c["tiene_disco"]), None)
if nodo_disco is None:
    raise RuntimeError("Ningún nodo tiene el disco montado.")
print(f"\nNodo fijado: {nodo_disco}")

ESTRATEGIA = NodeAffinitySchedulingStrategy(node_id=nodo_disco, soft=False)

archivos = ray.get(listar.options(scheduling_strategy=ESTRATEGIA).remote(RUTA))
print(f"{len(archivos)} archivos encontrados")

In [ ]:
resultados = ray.get([
    procesar.options(scheduling_strategy=ESTRATEGIA).remote(RUTA, a, SAT_THRESH, DEAD_THRESH, FS)
    for a in archivos
])

res_ok  = [r for r in resultados if "error" not in r]
res_err = [r for r in resultados if "error" in r]

meta = pd.DataFrame(res_ok).sort_values(["participante", "estimulo"]).reset_index(drop=True)

BANDS = {"Delta":(1,4), "Theta":(4,8), "Alpha":(8,13), "Beta":(13,30), "Gamma":(30,45)}
band_rows = []
for r in res_ok:
    row = {"participante": r["participante"], "estimulo": r["estimulo"]}
    if r["freqs"]:
        freqs, p = np.array(r["freqs"]), np.array(r["psd"])
        for b, (lo, hi) in BANDS.items():
            mask = (freqs >= lo) & (freqs <= hi)
            row[b] = float(np.trapz(p[mask], freqs[mask])) if mask.any() else np.nan
    band_rows.append(row)
band_df = pd.DataFrame(band_rows)

print(f"OK: {len(res_ok)}  errores: {len(res_err)}")
print(f"Canales muertos — media por trial: {meta['n_dead_ch'].mean():.1f} / {meta['n_ch_total'].iloc[0]}")
if res_err:
    print("Errores:", res_err[:3])
meta[["participante","estimulo","duracion_s","n_dead_ch","n_buenos_ch","mean_amp"]].head(10)

In [ ]:
estims = sorted(meta["estimulo"].unique())
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

# 1. Heatmap duración
ax = axes[0, 0]
piv = meta.pivot_table(index="participante", columns="estimulo", values="duracion_s")
im = ax.imshow(piv.values, aspect="auto", cmap="YlOrBr")
ax.set_xticks(range(len(piv.columns))); ax.set_xticklabels(piv.columns, fontsize=7)
ax.set_yticks(range(len(piv.index)));   ax.set_yticklabels([f"P{p}" for p in piv.index], fontsize=6)
plt.colorbar(im, ax=ax); ax.set_title("Duración (s)")

# 2. Heatmap canales saturados
ax = axes[0, 1]
piv2 = meta.pivot_table(index="participante", columns="estimulo", values="n_sat_ch")
im2 = ax.imshow(piv2.values, aspect="auto", cmap="RdYlGn_r")
ax.set_xticks(range(len(piv2.columns))); ax.set_xticklabels(piv2.columns, fontsize=7)
ax.set_yticks(range(len(piv2.index)));   ax.set_yticklabels([f"P{p}" for p in piv2.index], fontsize=6)
plt.colorbar(im2, ax=ax); ax.set_title("Canales saturados")

# 3. Completitud
ax = axes[0, 2]
comp  = meta.groupby("participante")["estimulo"].nunique()
n_tot = len(estims)
ax.bar([f"P{p}" for p in comp.index], comp.values,
       color=["tomato" if v < n_tot else "steelblue" for v in comp.values])
ax.axhline(n_tot, color="k", ls="--", lw=1, label=f"Total={n_tot}")
ax.legend(fontsize=7); ax.set_title("Estímulos por participante")
ax.tick_params(axis="x", rotation=90, labelsize=6)

# 4. Boxplot duración por estímulo
ax = axes[1, 0]
data = [meta[meta["estimulo"] == e]["duracion_s"].values for e in estims]
ax.boxplot(data, labels=estims, showfliers=False, patch_artist=True,
           medianprops=dict(color="red", lw=1.5))
ax.set_xlabel("Estímulo"); ax.set_ylabel("s"); ax.set_title("Duración por estímulo")

# 5. Alpha/Beta ratio
ax = axes[1, 1]
band_df["a_b"] = band_df["Alpha"] / band_df["Beta"].replace(0, np.nan)
data_ab = [band_df[band_df["estimulo"] == e]["a_b"].dropna().values for e in estims]
ax.boxplot(data_ab, labels=estims, showfliers=False, patch_artist=True,
           medianprops=dict(color="navy", lw=1.5))
ax.set_xlabel("Estímulo"); ax.set_ylabel("Alpha/Beta"); ax.set_title("Alpha/Beta (↓ = arousal)")

# 6. PSD media por estímulo (sobre señal filtrada)
ax = axes[1, 2]
psd_by_e = {}
for r in res_ok:
    if r["freqs"]:
        psd_by_e.setdefault(r["estimulo"], []).append(np.array(r["psd"]))
cmap = plt.cm.tab10(np.linspace(0, 1, max(len(psd_by_e), 1)))
freqs_ref = np.array(next(r["freqs"] for r in res_ok if r["freqs"]))
for i, (e, psds) in enumerate(sorted(psd_by_e.items())):
    ax.semilogy(freqs_ref, np.mean(psds, axis=0), color=cmap[i % len(cmap)], lw=1.2, label=f"Est {e}")
ax.set_xlim(0, 50); ax.set_xlabel("Hz"); ax.set_ylabel("µV²/Hz")
ax.set_title("PSD media por estímulo (filtrada 1–45 Hz)"); ax.legend(fontsize=6)

plt.suptitle("EDA — EEG por Estímulo Emocional", fontsize=13)
plt.tight_layout()
plt.savefig("eda_estimulos.png", dpi=150, bbox_inches="tight")
plt.show()
print("Guardado: eda_estimulos.png")

In [6]:
print(f"Participantes : {meta['participante'].nunique()}")
print(f"Estímulos     : {meta['estimulo'].nunique()} → {estims}")
print(f"Archivos OK   : {len(meta)}")
print(f"Duración media: {meta['duracion_s'].mean():.1f} s (±{meta['duracion_s'].std():.1f})")
print(f"Con saturación: {(meta['n_sat_ch'] > 0).sum()} archivos")
print()
print("Potencia mediana por banda:")
for b in BANDS:
    if b in band_df.columns:
        print(f"  {b:7s}: {band_df[b].median():.3e} µV²")

Participantes : 7
Estímulos     : 11 → [1, 3, 6, 10, 11, 12, 13, 14, 20, 26, 27]
Archivos OK   : 16
Duración media: 4.9 s (±0.0)
Con saturación: 15 archivos

Potencia mediana por banda:
  Delta  : 4.133e+02 µV²
  Theta  : 3.805e+01 µV²
  Alpha  : 1.966e+01 µV²
  Beta   : 4.324e+04 µV²
  Gamma  : 1.191e+01 µV²
